In [6]:
# minimal test

import pylink

jlink = pylink.JLink()

for probe in jlink.connected_emulators():
    print("J-Link detected")
    print("Serial:", probe.SerialNumber)
    

J-Link detected
Serial: 600105462


In [9]:
import pylink

jlink = pylink.JLink()

probes = jlink.connected_emulators()

if not probes:
    print("FAIL: No J-Link found")
else:
    serial = probes[0].SerialNumber
    print(f"OK: J-Link found: {serial}")

    try:
        jlink.open(serial_no=serial)
        print("OK: J-Link USB communication works")

        jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)
        jlink.connect("ATSAM4S2C", speed=1000)

        print("OK: SAM4S SWD connection works")
        #print(f"Target voltage: {jlink.target_voltage():.3f} V")

    except Exception as e:
        print("FAIL:", repr(e))

    finally:
        if jlink.opened():
            jlink.close()


OK: J-Link found: 600105462
OK: J-Link USB communication works
OK: SAM4S SWD connection works


In [8]:
import pylink

SERIAL = 600105462
DEVICE = "ATSAM4S2C"

jlink = pylink.JLink()

try:
    print("Opening J-Link...")
    jlink.open(serial_no=SERIAL)

    print("S/N:", jlink.serial_number)

    print(f"\nConnecting to {DEVICE} using SWD...")

    jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)
    jlink.connect(DEVICE)

    print("Connected:", jlink.connected())

    # -------------------------------------------------
    # Halt CPU before reading CPU registers
    # -------------------------------------------------

    print("\nHalting CPU...")
    jlink.halt()

    print("CPU halted:", jlink.halted())

    # Cortex-M registers
    r0 = jlink.register_read(0)
    sp = jlink.register_read(13)
    lr = jlink.register_read(14)
    pc = jlink.register_read(15)

    print()
    print(f"R0 : 0x{r0:08X}")
    print(f"SP : 0x{sp:08X}")
    print(f"LR : 0x{lr:08X}")
    print(f"PC : 0x{pc:08X}")

    # -------------------------------------------------
    # Read Cortex-M CPUID
    # -------------------------------------------------

    CPUID_ADDR = 0xE000ED00

    cpuid = jlink.memory_read32(CPUID_ADDR, 1)[0]

    print()
    print(f"CPUID: 0x{cpuid:08X}")

    print("\nJ-Link/SWD test PASSED")

    # -------------------------------------------------
    # Continue firmware execution
    # -------------------------------------------------

    print("\nRestarting CPU...")
    jlink.restart()

    print("CPU halted:", jlink.halted())

except Exception as e:
    print()
    print("J-Link test FAILED")
    print(type(e).__name__ + ":", e)

finally:
    jlink.close()
    print("\nJ-Link closed.")

Opening J-Link...
S/N: 600105462

Connecting to ATSAM4S2C using SWD...
Connected: True

Halting CPU...
CPU halted: True

R0 : 0x00000001
SP : 0x20001088
LR : 0x00400C59
PC : 0x00400C50

CPUID: 0x410FC241

J-Link/SWD test PASSED

Restarting CPU...
CPU halted: False

J-Link closed.


In [11]:
import pylink
import sys
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DEVICE = "ATSAM4S2A"

FLASH_START = 0x00400000
FLASH_SIZE  = 256 * 1024       # 128 KiB

BLOCK_SIZE = 1024

OUTPUT_FILE = "sam4s_flash.bin"


def main():
    jlink = pylink.JLink()

    try:
        # ----------------------------------------------------
        # Find J-Link
        # ----------------------------------------------------

        probes = jlink.connected_emulators()

        if not probes:
            raise RuntimeError("No J-Link probe detected")

        print("Detected J-Link probes:")

        for probe in probes:
            print(f"  S/N: {probe.SerialNumber}")

        serial = probes[0].SerialNumber

        # ----------------------------------------------------
        # Open J-Link
        # ----------------------------------------------------

        print()
        print(f"Opening J-Link S/N {serial}...")

        jlink.open(serial_no=serial)

        # ----------------------------------------------------
        # Connect to SAM4S
        # ----------------------------------------------------

        jlink.set_tif(
            pylink.enums.JLinkInterfaces.SWD
        )

        print(f"Connecting to {DEVICE}...")

        jlink.connect(
            DEVICE,
            speed=1000
        )

        print("Connected.")
        #print(f"Target voltage: {jlink.target_voltage():.3f} V")

        # ----------------------------------------------------
        # Read flash
        # ----------------------------------------------------

        print()
        print(
            f"Reading flash:"
            f" 0x{FLASH_START:08X}"
            f" - 0x{FLASH_START + FLASH_SIZE - 1:08X}"
        )

        flash = bytearray()

        for offset in range(0, FLASH_SIZE, BLOCK_SIZE):

            address = FLASH_START + offset

            length = min(
                BLOCK_SIZE,
                FLASH_SIZE - offset
            )

            data = jlink.memory_read8(
                address,
                length
            )

            flash.extend(data)

            percent = (
                (offset + length)
                / FLASH_SIZE
                * 100
            )

            print(
                f"\r0x{address:08X}: "
                f"{offset + length:6d}/{FLASH_SIZE} "
                f"({percent:5.1f}%)",
                end="",
                flush=True
            )

        print()
        print()

        # ----------------------------------------------------
        # Save binary
        # ----------------------------------------------------

        Path(OUTPUT_FILE).write_bytes(flash)

        print(f"Saved {len(flash)} bytes")
        print(f"File: {OUTPUT_FILE}")

        # ----------------------------------------------------
        # Show first 64 bytes
        # ----------------------------------------------------

        print()
        print("First 64 bytes:")

        for offset in range(0, min(64, len(flash)), 16):

            chunk = flash[offset:offset + 16]

            hexdata = " ".join(
                f"{b:02X}" for b in chunk
            )

            print(
                f"{FLASH_START + offset:08X}: "
                f"{hexdata}"
            )

    except Exception as e:

        print()
        print("ERROR:")
        print(type(e).__name__, e)

        sys.exit(1)

    finally:

        try:
            if jlink.opened():
                jlink.close()
        except Exception:
            pass


if __name__ == "__main__":
    main()
    

Detected J-Link probes:
  S/N: 600105462

Opening J-Link S/N 600105462...
Connecting to ATSAM4S2A...
Connected.

Reading flash: 0x00400000 - 0x0043FFFF
0x0043FC00: 262144/262144 (100.0%)

Saved 262144 bytes
File: sam4s_flash.bin

First 64 bytes:
00400000: C8 11 00 20 BD 0A 40 00 B9 0A 40 00 B9 0A 40 00
00400010: B9 0A 40 00 B9 0A 40 00 B9 0A 40 00 00 00 00 00
00400020: 00 00 00 00 00 00 00 00 00 00 00 00 B9 0A 40 00
00400030: B9 0A 40 00 00 00 00 00 B9 0A 40 00 B9 0A 40 00
